In [ ]:
-- Welcome to your new notebook
-- Type here in the cell editor to add code!

-- from pyspark.sql.functions import *
-- from pyspark.sql import Row

In [2]:
-- Product table (combination of products and prod_category)

CREATE  TABLE Retail_Gold.dbo.products AS 
WITH joined_data AS (
  SELECT 
    p.product_id, 
    p.photo_qty, 
    p.weight, 
    p.length, 
    p.width, 
    p.height, 
    c.product_category_name_english as category_name
  FROM Retail_Silver.dbo.products AS p
  JOIN Retail_Silver.dbo.prod_category AS c 
    ON p.category = c.product_category_name
)
SELECT 
  product_id, 
  category_name, 
  photo_qty, 
  weight, 
  COALESCE(length * width * height) AS volume
FROM joined_data;



StatementMeta(, 2f34ceb2-a971-4f1d-abec-457678d38103, 3, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [5]:
select count(*) from Retail_Lakehouse.dbo.olist_order_items_dataset

StatementMeta(, 2f34ceb2-a971-4f1d-abec-457678d38103, 6, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [1]:
CREATE OR REPLACE TABLE Retail_Gold.dbo.items AS 
SELECT 
    order_id,
    item_id,
    product_id,
    seller_id,
    shipping_limit_date,
    CAST(price AS DECIMAL(10, 1)) AS price,
    CAST(freight_value AS DECIMAL(10, 1)) AS freight_value
FROM Retail_Silver.dbo.items;


StatementMeta(, 3ccccec2-4216-46df-9a2b-cf5f0a0065f8, 2, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [2]:
CREATE OR REPLACE TABLE Retail_Gold.dbo.customers AS
SELECT customer_id as order_customer_id, customer_unique_id as customer_id, COALESCE(zip_code, 0) AS zip_code, COALESCE(city,'unknown') AS city, COALESCE(state, 'unknown') As state
FROM Retail_Silver.dbo.customers
WHERE customer_id IS NOT NULL AND custoer_unique_id IS NOT NULL

StatementMeta(, 723e5f72-1941-46c5-9238-ebf1c0519187, 3, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [8]:
CREATE OR REPLACE TABLE Retail_Gold.dbo.location AS
SELECT *
FROM Retail_Silver.dbo.location 

StatementMeta(, 723e5f72-1941-46c5-9238-ebf1c0519187, 9, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [9]:
CREATE OR REPLACE TABLE Retail_Gold.dbo.reviews AS
SELECT review_id, order_id, coalesce(review_score,0) as review_score, coalesce(review_comment_title, 'No title') as review_title, coalesce(review_comment_message, 'No message') as review_message
from Retail_Silver.dbo.reviews

StatementMeta(, 723e5f72-1941-46c5-9238-ebf1c0519187, 10, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [1]:
CREATE OR REPLACE TABLE Retail_Gold.dbo.orders AS
SELECT order_id, customer_id as order_customer_id, order_status, purchase_date, delivery_date, estimated_delivery_date
from Retail_Silver.dbo.orders


StatementMeta(, fa629931-bf4f-440d-9aa6-8e99f211d2fb, 2, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [1]:
CREATE OR REPLACE TABLE Retail_Gold.dbo.payments
select order_id, payment_sequential, payment_type, CAST((installments * value) AS DECIMAL(10, 1)) as total --, ARRAY_AGG( payment_type) AS payment_types_used
from Retail_Silver.dbo.payments


StatementMeta(, 53a17422-af96-4175-84b0-725e15f84bd7, 2, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [1]:
%%sql
-- Clear the active session cache for the payments table
--REFRESH TABLE Retail_Silver.dbo.payments;

-- Re-run your duplicate type check query
CREATE OR REPLACE TABLE Retail_Gold.dbo.sellers AS
SELECT *
FROM Retail_Silver.dbo.sellers
WHERE seller_id IS NOT NULL


StatementMeta(, f9fe3651-3b46-48f8-8944-67056e1f6abe, 2, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [3]:
-- Refresh cached table metadata
--REFRESH TABLE Retail_Silver.dbo.payments;

-- Identify orders with multiple distinct payment types
SELECT 
    order_id,
    payment_sequential,
    COUNT(DISTINCT payment_type) AS unique_payment_count,
    ARRAY_AGG(DISTINCT payment_type) AS payment_types_used
FROM Retail_Silver.dbo.payments
GROUP BY order_id
HAVING COUNT(DISTINCT payment_type) > 1;


StatementMeta(, a1a3ac4f-e620-4543-a5c3-2099af61e04d, 5, Finished, Available, Finished, False)

<Spark SQL result set with 1000 rows and 3 fields>

In [2]:
%%sql
SELECT * FROM Retail_Silver.dbo.payments 
WHERE order_id = '8ca5bdac5ebe8f2d6fc9171d5ebc906a'

StatementMeta(, fe943cb9-a472-4e47-9b63-38721db49240, 3, Finished, Available, Finished, False)

<Spark SQL result set with 9 rows and 5 fields>